# YOLO two-stage heuristic data preparation pipeline

This notebook processes `train`, `valid`, and `test` from `/work/data/classification_dataset/`, filters images by selected COCO category labels, runs the same two-stage YOLO heuristic used in `yolo_lightning_test_v2.ipynb`, saves cropped tooth images, and adds extra crops for annotated teeth that YOLO missed.

Run the cells from top to bottom. The configuration cell is intentionally centralized so labels, thresholds, margins, and paths can be changed safely.

In [58]:
%matplotlib inline

## 1. Setup imports and paths

In [59]:
from __future__ import annotations

import os
import sys
import json
import uuid
import math
import shutil
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Iterable
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TF
from PIL import Image
from tqdm.auto import tqdm

# Required workspace paths from the task description.
PROJECT_ROOT = Path('/work')
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'
CLASSIFICATION_DATASET_ROOT = Path('/work/data/classification_dataset')
NOTEBOOK_OUTPUT_DIR = Path('/work/scripts/detection_pipeline/inference pipelines/dataprep')

# Make project imports work when this notebook is saved/run from NOTEBOOK_OUTPUT_DIR.
for p in (PROJECT_ROOT, SCRIPTS_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Classification dataset root:', CLASSIFICATION_DATASET_ROOT)
print('Notebook/output working directory:', NOTEBOOK_OUTPUT_DIR)


Device: cuda
Classification dataset root: /work/data/classification_dataset
Notebook/output working directory: /work/scripts/detection_pipeline/inference pipelines/dataprep


## 2. Configuration

`SELECTED_IMAGE_CATEGORY_NAMES` controls which COCO labels are accepted for processing. Run the category-inspection cell below before changing it if you are unsure about the exact names in your annotation files.

The default is deliberately broad and accepts category names that contain `frontal`. You can replace it with exact names such as `['frontal']`, `['Frontal radiograph']`, or whatever appears in your COCO files.

In [60]:
@dataclass
class PipelineConfig:
    # Input/output.
    classification_dataset_root: Path = CLASSIFICATION_DATASET_ROOT
    output_root: Path = NOTEBOOK_OUTPUT_DIR / 'prepared_tooth_crops'
    splits: tuple[str, ...] = ('train', 'valid', 'test')
    coco_filename: str = '_annotations.coco.json'

    # User-selectable image filtering labels.
    # If empty, all images are processed. Otherwise, an image is processed when it has
    # at least one COCO annotation whose category name/id matches these selections.
    selected_image_category_names: tuple[str, ...] = ()

    selected_image_category_ids: tuple[int, ...] = (#1, #34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51,
                                                    #52 , 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70,
                                                    #71, 72, 73, 74, 75, 76, 77, 78, 79)
    )
    category_name_match_mode: str = 'startswith'  # 'contains' or 'exact', case-insensitive

    # YOLO/model configuration copied from the evaluation notebook.
    image_size: int = 640
    experiment_name: str = 'label-loss-fixed2'
    pretrained_weights: str = 'yolov5s.pt'
    num_classes: int = 32

    # Raw YOLO inference thresholds for the two-stage heuristic.
    heuristic_conf_threshold: float = 0.29
    heuristic_iou_threshold: float = 0.45
    heuristic_max_detections: int = 200
    inter_class_nms_threshold: float = 0.75

    # The notebook's classification-image inference used a pre-inference zoom/crop:
    # keep 90% of the image centered horizontally and around the bottom 2/3 vertically.
    use_pre_inference_zoom_crop: bool = True
    zoom_factor: float = 0.90
    zoom_center_x_fraction: float = 0.50
    zoom_center_y_fraction: float = 2 / 3

    # Crop settings.
    yolo_crop_margin: float = 0.00      # set >0 if you want YOLO-predicted crops expanded too
    missed_gt_crop_margin: float = 0.08 # required by the task; configurable

    # Matching GT annotations to YOLO predictions for the "missed teeth" step.
    missed_match_iou_threshold: float = 0.50

    # YOLO labels are normally zero-based after non_max_suppression.
    # COCO category ids are often one-based. Keep 1 if YOLO class 0 corresponds to COCO category_id 1.
    yolo_label_to_coco_category_id_offset: int = 1

    # File-writing behavior.
    overwrite_existing_run: bool = False
    jpeg_quality: int = 95

cfg = PipelineConfig()

RUN_ID = uuid.uuid4().hex[:10]
RUN_OUTPUT_DIR = cfg.output_root / RUN_ID
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=cfg.overwrite_existing_run)

print('Run ID:', RUN_ID)
print('Run output directory:', RUN_OUTPUT_DIR)


Run ID: 4ef020aad4
Run output directory: /work/scripts/detection_pipeline/inference pipelines/dataprep/prepared_tooth_crops/4ef020aad4


## 3. Load and inspect COCO metadata for all splits

In [61]:
def load_coco_for_split(split: str) -> dict[str, Any]:
    ann_path = cfg.classification_dataset_root / split / cfg.coco_filename
    if not ann_path.exists():
        raise FileNotFoundError(f'Missing COCO annotation file: {ann_path}')
    with ann_path.open('r', encoding='utf-8') as f:
        return json.load(f)

coco_by_split = {split: load_coco_for_split(split) for split in cfg.splits}

all_categories = {}
for split, coco in coco_by_split.items():
    for cat in coco.get('categories', []):
        all_categories[int(cat['id'])] = cat.get('name', str(cat['id']))

categories_df = pd.DataFrame(
    [{'category_id': cid, 'category_name': name} for cid, name in sorted(all_categories.items())]
)
print(f'Loaded COCO files for splits: {list(coco_by_split)}')
display(categories_df)
categories_txt_path = RUN_OUTPUT_DIR / 'categories.txt'
categories_df.to_csv(categories_txt_path, index=False, sep='\t')
print(f'Saved category mappings to: {categories_txt_path}')


Loaded COCO files for splits: ['train', 'valid', 'test']


,category_id,category_name
0,0,see
1,1,3M ESPE Implant
2,2,47
3,3,48
4,4,49
...,...,...
75,75,Root canal obturation
76,76,Sterngold Implant
77,77,Straumann Implant
78,78,Titan Implant Implant


Saved category mappings to: /work/scripts/detection_pipeline/inference pipelines/dataprep/prepared_tooth_crops/4ef020aad4/categories.txt


## 4. Build image/annotation records and filter frontal radiographs

In [62]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def category_name_selected(name: str) -> bool:
    if not cfg.selected_image_category_names:
        return True
    
    name_l = str(name).lower()
    selected = [s.lower() for s in cfg.selected_image_category_names]
    
    if cfg.category_name_match_mode == 'exact':
        return name_l in selected
    if cfg.category_name_match_mode == 'contains':
        return any(s in name_l for s in selected)
        
    # ÚJ LOGIKA: Ha a kategória neve a megadott szavakkal kezdődik
    if cfg.category_name_match_mode == 'startswith':
        return any(name_l.startswith(s) for s in selected)
        
    raise ValueError("category_name_match_mode must be 'contains', 'exact', or 'startswith'")


def category_id_selected(category_id: int, id_to_name: dict[int, str]) -> bool:
    if cfg.selected_image_category_ids and int(category_id) in set(cfg.selected_image_category_ids):
        return True
    if cfg.selected_image_category_names:
        return category_name_selected(id_to_name.get(int(category_id), str(category_id)))
    return True


def resolve_image_path(split_dir: Path, file_name: str) -> Path:
    direct = split_dir / file_name
    if direct.exists():
        return direct
    matches = list(split_dir.rglob(Path(file_name).name))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not resolve image file {file_name!r} under {split_dir}')


def build_split_records(split: str, coco: dict[str, Any]) -> list[dict[str, Any]]:
    split_dir = cfg.classification_dataset_root / split
    id_to_name = {int(c['id']): c.get('name', str(c['id'])) for c in coco.get('categories', [])}
    images_by_id = {int(img['id']): img for img in coco.get('images', [])}
    anns_by_image_id = defaultdict(list)
    for ann in coco.get('annotations', []):
        anns_by_image_id[int(ann['image_id'])].append(ann)

    records = []
    for image_id, image_info in images_by_id.items():
        raw_file_name = image_info['file_name']
        
        # --- ÚJ, BIZTONSÁGOS LOGIKA A FÁJLNEVEKRE ---
        # A Path().name gondoskodik róla, hogy ha a file_name "train/cate2-0089.jpg",
        # akkor is csak a "cate2-0089.jpg" részt vizsgálja!
        just_file_name = Path(raw_file_name).name.lower()
        
        if not just_file_name.startswith('cate'):
            continue
        # --------------------------------------------

        anns = anns_by_image_id.get(image_id, [])
        
        # FIGYELEM: A kategória (fogak/implantátumok) szerinti szűrést INNEN TELJESEN KIVETTEM.
        # Ha a fájlnév 'cate'-tel kezdődik, mindenképp feldolgozzuk a képet.

        image_path = resolve_image_path(split_dir, raw_file_name)
        records.append({
            'split': split,
            'image_id': image_id,
            'file_name': raw_file_name,
            'image_path': image_path,
            'width': image_info.get('width'),
            'height': image_info.get('height'),
            'annotations': anns,
            'id_to_name': id_to_name,
        })
    return records

# Újrafuttatjuk a rekordok összegyűjtését
records_by_split = {split: build_split_records(split, coco) for split, coco in coco_by_split.items()}
all_records = [r for split_records in records_by_split.values() for r in split_records]

summary_df = pd.DataFrame([
    {'split': split, 'filtered_images': len(records_by_split[split]), 'all_images': len(coco_by_split[split].get('images', []))}
    for split in cfg.splits
])
display(summary_df)
all_records = all_records[:5]
print(f'Total filtered images to process: {len(all_records)}')



,split,filtered_images,all_images
0,train,1261,6263
1,valid,0,1342
2,test,265,1342


Total filtered images to process: 5


## 5. Load YOLOv5 code and model weights

In [63]:
# Clone YOLOv5 if missing and put it first in import order for YOLO-specific utils.
YOLOV5_DIR = PROJECT_ROOT / 'external' / 'yolov5'
YOLOV5_DIR.parent.mkdir(parents=True, exist_ok=True)

if not YOLOV5_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', 'https://github.com/ultralytics/yolov5.git', str(YOLOV5_DIR)
    ], check=True)

if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))

from models.yolo import Model
from utils.general import non_max_suppression

print('YOLOv5 code loaded from:', YOLOV5_DIR)


YOLOv5 code loaded from: /work/external/yolov5


In [64]:
class LitYOLOv5(torch.nn.Module):
    """Minimal wrapper compatible with the model-loading logic used in yolo_lightning_test_v2.ipynb."""

    def __init__(self, image_size: int, pretrained_weights: str, num_classes: int = 32):
        super().__init__()
        self.image_size = image_size
        self.num_classes = num_classes

        ckpt_path = Path(pretrained_weights)
        if ckpt_path.exists():
            ckpt = torch.load(ckpt_path, map_location='cpu')
            yolo_cfg = ckpt['model'].yaml
        else:
            yolo_cfg = YOLOV5_DIR / 'models' / 'yolov5s.yaml'

        self.model = Model(yolo_cfg, ch=3, nc=num_classes).float()
        self.model.hyp = {
            'box': 0.05, 'cls': 0.3, 'obj': 0.7,
            'cls_pw': 1.0, 'obj_pw': 1.0, 'fl_gamma': 0.0,
            'label_smoothing': 0.0, 'anchor_t': 4.0,
        }

model_path = PROJECT_ROOT / 'final_models' / cfg.experiment_name / 'weights' / 'final_model.pt'
if not model_path.exists():
    raise FileNotFoundError(f'Model weights not found at {model_path}')

print(f'Loading YOLO weights from {model_path}...')
model_instance = LitYOLOv5(
    image_size=cfg.image_size,
    pretrained_weights=cfg.pretrained_weights,
    num_classes=cfg.num_classes,
)
model_instance.model.load_state_dict(torch.load(model_path, map_location='cpu'), strict=False)
model_instance = model_instance.to(DEVICE).eval()
print('Model ready.')


Overriding model.yaml nc=80 with nc=32

                 from  n    params  module                                  arguments                     
  0                -1  1      3520  models.common.Conv                      [3, 32, 6, 2, 2]              


  1                -1  1     18560  models.common.Conv                      [32, 64, 3, 2]                
  2                -1  1     18816  models.common.C3                        [64, 64, 1]                   
  3                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  4                -1  2    115712  models.common.C3                        [128, 128, 2]                 
  5                -1  1    295424  models.common.Conv                      [128, 256, 3, 2]              
  6                -1  3    625152  models.common.C3                        [256, 256, 3]                 
  7                -1  1   1180672  models.common.Conv                      [256, 512, 3, 2]              
  8                -1  1   1182720  models.common.C3                        [512, 512, 1]                 
  9                -1  1    656896  models.common.SPPF                      [512, 512, 5]                 
 10                -1  1    131584  m

Loading YOLO weights from /work/final_models/label-loss-fixed2/weights/final_model.pt...


YOLOv5s summary: 214 layers, 7105933 parameters, 7105933 gradients, 16.2 GFLOPs



Model ready.


## 6. Inference preprocessing and coordinate mapping helpers

In [65]:
def compute_zoom_crop_box(width: int, height: int) -> tuple[int, int, int, int]:
    if not cfg.use_pre_inference_zoom_crop:
        return 0, 0, width, height

    crop_w = int(width * cfg.zoom_factor)
    crop_h = int(height * cfg.zoom_factor)
    center_x = int(width * cfg.zoom_center_x_fraction)
    center_y = int(height * cfg.zoom_center_y_fraction)

    x1 = max(0, center_x - crop_w // 2)
    y1 = max(0, center_y - crop_h // 2)
    x2 = min(width, x1 + crop_w)
    y2 = min(height, y1 + crop_h)

    if x2 == width:
        x1 = max(0, width - crop_w)
    if y2 == height:
        y1 = max(0, height - crop_h)

    return int(x1), int(y1), int(x2), int(y2)


def prepare_image_for_yolo(orig_img: Image.Image) -> tuple[torch.Tensor, dict[str, Any]]:
    """Apply the notebook's zoom crop, letterbox resize, and padding."""
    orig_w, orig_h = orig_img.size
    zx1, zy1, zx2, zy2 = compute_zoom_crop_box(orig_w, orig_h)
    inference_img = orig_img.crop((zx1, zy1, zx2, zy2))

    inf_w, inf_h = inference_img.size
    scale = cfg.image_size / max(inf_w, inf_h)
    new_w, new_h = int(inf_w * scale), int(inf_h * scale)

    resized = inference_img.resize((new_w, new_h), Image.Resampling.BILINEAR)
    pad_w = cfg.image_size - new_w
    pad_h = cfg.image_size - new_h
    padded = TF.pad(resized, (0, 0, pad_w, pad_h), fill=0)
    tensor = TF.to_tensor(padded)

    meta = {
        'orig_w': orig_w,
        'orig_h': orig_h,
        'zoom_box': (zx1, zy1, zx2, zy2),
        'inference_w': inf_w,
        'inference_h': inf_h,
        'scale': scale,
        'new_w': new_w,
        'new_h': new_h,
        'pad_w': pad_w,
        'pad_h': pad_h,
    }
    return tensor, meta


def yolo_boxes_to_original_xyxy(boxes: torch.Tensor, meta: dict[str, Any]) -> torch.Tensor:
    """Map boxes from padded cfg.image_size space back to original-image coordinates."""
    if len(boxes) == 0:
        return boxes.clone().float()

    boxes = boxes.detach().cpu().float().clone()
    scale = float(meta['scale'])
    zx1, zy1, zx2, zy2 = meta['zoom_box']
    inf_w, inf_h = int(meta['inference_w']), int(meta['inference_h'])
    orig_w, orig_h = int(meta['orig_w']), int(meta['orig_h'])

    boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, meta['new_w']) / scale + zx1
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, meta['new_h']) / scale + zy1

    boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, orig_w)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, orig_h)
    return boxes


## 7. Two-stage heuristic helpers

In [66]:
def box_iou_xyxy(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    if len(boxes1) == 0 or len(boxes2) == 0:
        return torch.zeros((len(boxes1), len(boxes2)), device=boxes1.device)

    x1 = torch.max(boxes1[:, None, 0], boxes2[None, :, 0])
    y1 = torch.max(boxes1[:, None, 1], boxes2[None, :, 1])
    x2 = torch.min(boxes1[:, None, 2], boxes2[None, :, 2])
    y2 = torch.min(boxes1[:, None, 3], boxes2[None, :, 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
    union = area1[:, None] + area2[None, :] - inter
    return inter / union.clamp(min=1e-6)


def apply_one_per_class_with_indices(pred: dict[str, torch.Tensor]) -> tuple[dict[str, torch.Tensor], set[int]]:
    labels = pred['labels'].detach().cpu()
    scores = pred['scores'].detach().cpu()

    keep_indices = []
    for label in torch.unique(labels):
        class_indices = torch.where(labels == label)[0]
        best_local = torch.argmax(scores[class_indices])
        keep_indices.append(int(class_indices[best_local]))

    keep_indices = sorted(keep_indices)
    return {
        'boxes': pred['boxes'][keep_indices],
        'scores': pred['scores'][keep_indices],
        'labels': pred['labels'][keep_indices],
    }, set(keep_indices)


def apply_two_stage_heuristic_with_logging(
    pred: dict[str, torch.Tensor],
    conflict_iou_threshold: float | None = None,
) -> tuple[dict[str, torch.Tensor], list[dict[str, Any]]]:
    """Stage 1: keep highest-confidence prediction per class. Stage 2: cross-class IoU suppression."""
    if conflict_iou_threshold is None:
        conflict_iou_threshold = cfg.inter_class_nms_threshold

    heur_pred, _ = apply_one_per_class_with_indices(pred)
    boxes = heur_pred['boxes']
    scores = heur_pred['scores']
    labels = heur_pred['labels']

    if len(boxes) <= 1:
        return heur_pred, []

    order = torch.argsort(scores, descending=True)
    final_keep = []
    removed_info = []

    for idx in order.tolist():
        if not final_keep:
            final_keep.append(idx)
            continue

        current_box = boxes[idx].unsqueeze(0)
        kept_boxes = boxes[final_keep]
        ious = box_iou_xyxy(current_box, kept_boxes)[0]
        max_iou = torch.max(ious)

        if max_iou <= conflict_iou_threshold:
            final_keep.append(idx)
        else:
            best_kept_local_idx = torch.argmax(ious).item()
            best_kept_idx = final_keep[best_kept_local_idx]
            removed_info.append({
                'removed_box': boxes[idx].detach().cpu().numpy().tolist(),
                'removed_label': int(labels[idx].item()),
                'removed_score': float(scores[idx].item()),
                'kept_box': boxes[best_kept_idx].detach().cpu().numpy().tolist(),
                'kept_label': int(labels[best_kept_idx].item()),
                'kept_score': float(scores[best_kept_idx].item()),
                'iou': float(max_iou.item()),
            })

    final_keep = sorted(final_keep)
    return {
        'boxes': boxes[final_keep],
        'scores': scores[final_keep],
        'labels': labels[final_keep],
    }, removed_info


@torch.no_grad()
def run_yolo_inference_on_images(
    model: LitYOLOv5,
    images: torch.Tensor,
    *,
    conf_threshold: float | None = None,
    iou_threshold: float | None = None,
    max_det: int | None = None,
) -> list[dict[str, torch.Tensor]]:
    conf_threshold = cfg.heuristic_conf_threshold if conf_threshold is None else conf_threshold
    iou_threshold = cfg.heuristic_iou_threshold if iou_threshold is None else iou_threshold
    max_det = cfg.heuristic_max_detections if max_det is None else max_det

    model.eval()
    images = images.to(DEVICE).float()
    raw_output = model.model(images)
    preds = raw_output[0] if isinstance(raw_output, (tuple, list)) else raw_output

    nms_preds = non_max_suppression(
        preds,
        conf_thres=conf_threshold,
        iou_thres=iou_threshold,
        multi_label=False,
        max_det=max_det,
    )

    results = []
    for det in nms_preds:
        if det is None or len(det) == 0:
            results.append({
                'boxes': torch.zeros((0, 4)),
                'scores': torch.zeros((0,)),
                'labels': torch.zeros((0,), dtype=torch.long),
            })
        else:
            results.append({
                'boxes': det[:, :4].detach().cpu(),
                'scores': det[:, 4].detach().cpu(),
                'labels': det[:, 5].long().detach().cpu(),
            })
    return results


## 8. COCO bounding-box, cropping, and missed-GT helpers

In [67]:
def segmentation_to_bbox(segmentation: Any) -> list[float] | None:
    """Convert COCO polygon segmentation to [x, y, w, h]. RLE is not decoded here."""
    if not segmentation:
        return None

    xs, ys = [], []
    if isinstance(segmentation, list):
        for poly in segmentation:
            if not poly:
                continue
            arr = np.asarray(poly, dtype=float).reshape(-1, 2)
            xs.extend(arr[:, 0].tolist())
            ys.extend(arr[:, 1].tolist())

    if not xs or not ys:
        return None
    x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
    return [float(x1), float(y1), float(x2 - x1), float(y2 - y1)]


def annotation_to_xyxy(ann: dict[str, Any]) -> list[float] | None:
    bbox = ann.get('bbox')
    if not bbox:
        bbox = segmentation_to_bbox(ann.get('segmentation'))
    if not bbox:
        return None
    x, y, w, h = [float(v) for v in bbox]
    return [x, y, x + w, y + h]


def expand_and_clip_xyxy(
    box: Iterable[float],
    width: int,
    height: int,
    margin: float = 0.0,
) -> tuple[int, int, int, int] | None:
    x1, y1, x2, y2 = [float(v) for v in box]
    bw = max(0.0, x2 - x1)
    bh = max(0.0, y2 - y1)
    if bw <= 1 or bh <= 1:
        return None

    dx = bw * margin
    dy = bh * margin
    x1 = max(0, math.floor(x1 - dx))
    y1 = max(0, math.floor(y1 - dy))
    x2 = min(width, math.ceil(x2 + dx))
    y2 = min(height, math.ceil(y2 + dy))

    if x2 <= x1 or y2 <= y1:
        return None
    return int(x1), int(y1), int(x2), int(y2)


def safe_stem_for_image_folder(file_name: str) -> str:
    # Keep the original filename visible but avoid nested paths inside the output tree.
    return Path(file_name).name


def save_crop_with_metadata(
    *,
    orig_img: Image.Image,
    output_image_dir: Path,
    crop_box: tuple[int, int, int, int],
    crop_name: str,
    metadata: dict[str, Any],
) -> dict[str, Any]:
    output_image_dir.mkdir(parents=True, exist_ok=True)
    crop_path = output_image_dir / f'{crop_name}.jpg'

    crop = orig_img.crop(crop_box)
    crop.save(crop_path, quality=cfg.jpeg_quality)

    metadata = dict(metadata)
    metadata['crop_path'] = str(crop_path)
    metadata['crop_box_xyxy'] = list(map(int, crop_box))
    metadata['crop_width'] = int(crop_box[2] - crop_box[0])
    metadata['crop_height'] = int(crop_box[3] - crop_box[1])

    # INNEN KIVETTÜK A JSON MENTÉST! Csak visszaadjuk a dict-et.
    return metadata


def yolo_label_to_coco_category_id(yolo_label: int) -> int:
    return int(yolo_label) + int(cfg.yolo_label_to_coco_category_id_offset)


def find_missed_annotations(
    annotations: list[dict[str, Any]],
    pred_boxes_original: torch.Tensor,
    pred_labels: torch.Tensor,
) -> list[dict[str, Any]]:
    """Return annotated teeth not matched by a same-class YOLO prediction."""
    missed = []
    if len(annotations) == 0:
        return missed

    pred_boxes = pred_boxes_original.detach().cpu().float()
    pred_coco_category_ids = torch.tensor(
        [yolo_label_to_coco_category_id(int(x)) for x in pred_labels.detach().cpu().tolist()],
        dtype=torch.long,
    )

    for ann in annotations:
        gt_xyxy = annotation_to_xyxy(ann)
        if gt_xyxy is None:
            continue

        gt_category_id = int(ann['category_id'])
        same_class = torch.where(pred_coco_category_ids == gt_category_id)[0]
        is_matched = False

        if len(same_class) > 0:
            gt_box = torch.tensor(gt_xyxy, dtype=torch.float32).unsqueeze(0)
            ious = box_iou_xyxy(gt_box, pred_boxes[same_class])[0]
            is_matched = bool(torch.max(ious).item() >= cfg.missed_match_iou_threshold)

        if not is_matched:
            missed.append(ann)

    return missed


## 9. Optional sanity check on a few filtered images

In [68]:
preview_records = all_records[:5]
for r in preview_records:
    print(f"{r['split']} | image_id={r['image_id']} | {r['file_name']} | annotations={len(r['annotations'])}")

print('Change cfg.selected_image_category_names / ids above and rerun cells 4 onward if this selection is not correct.')


train | image_id=0 | cate2-00098_jpg.rf.84a05e227522684a3fbc382976efa9b4.jpg | annotations=32
train | image_id=1 | cate4-00001_jpg.rf.4783271f6cc246259775e4cdd0d49930.jpg | annotations=32
train | image_id=3 | cate2-00008_jpg.rf.9332bedf08202a2e53fbd5ebe6d23692.jpg | annotations=32
train | image_id=9 | cate8-00116_jpg.rf.65d6558c57178b29ed82f219a238341b.jpg | annotations=10
train | image_id=10 | cate4-00106_jpg.rf.0b0e2b3d9605d72a3dff19bd1598180e.jpg | annotations=32
Change cfg.selected_image_category_names / ids above and rerun cells 4 onward if this selection is not correct.


## 10. Run the full data-preparation pipeline

In [69]:
manifest_rows: list[dict[str, Any]] = []
run_config_path = RUN_OUTPUT_DIR / 'run_config.json'
with run_config_path.open('w', encoding='utf-8') as f:
    cfg_dict = asdict(cfg)
    cfg_dict = {k: str(v) if isinstance(v, Path) else v for k, v in cfg_dict.items()}
    json.dump({'run_id': RUN_ID, 'config': cfg_dict}, f, indent=2)

model_instance.eval()

for record in tqdm(all_records, desc='Processing filtered classification images'):
    split = record['split']
    image_path = Path(record['image_path'])
    original_filename = safe_stem_for_image_folder(record['file_name'])
    image_output_dir = RUN_OUTPUT_DIR / split / original_filename

    try:
        orig_img = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f'Could not open {image_path}: {e}')
        continue

    orig_w, orig_h = orig_img.size
    image_tensor, prep_meta = prepare_image_for_yolo(orig_img)

    raw_pred = run_yolo_inference_on_images(
        model_instance,
        image_tensor.unsqueeze(0),
        conf_threshold=cfg.heuristic_conf_threshold,
        iou_threshold=cfg.heuristic_iou_threshold,
        max_det=cfg.heuristic_max_detections,
    )[0]

    final_pred_padded, removed_info = apply_two_stage_heuristic_with_logging(
        raw_pred,
        conflict_iou_threshold=cfg.inter_class_nms_threshold,
    )

    pred_boxes_original = yolo_boxes_to_original_xyxy(final_pred_padded['boxes'], prep_meta)
    pred_labels = final_pred_padded['labels'].detach().cpu()
    pred_scores = final_pred_padded['scores'].detach().cpu()

    image_folder_metadata = []

    # Save YOLO-selected tooth crops.
    for i, (box, label, score) in enumerate(zip(pred_boxes_original, pred_labels, pred_scores)):
        crop_box = expand_and_clip_xyxy(box.tolist(), orig_w, orig_h, margin=cfg.yolo_crop_margin)
        if crop_box is None:
            continue

        yolo_label = int(label.item())
        coco_category_id = yolo_label_to_coco_category_id(yolo_label)
        category_name = record['id_to_name'].get(coco_category_id, str(coco_category_id))
        crop_name = f'yolo_{i:03d}_class_{coco_category_id}_score_{float(score):.3f}'.replace('.', 'p')

        meta = save_crop_with_metadata(
            orig_img=orig_img,
            output_image_dir=image_output_dir,
            crop_box=crop_box,
            crop_name=crop_name,
            metadata={
                'source': 'yolo_two_stage',
                'run_id': RUN_ID,
                'split': split,
                'original_image_path': str(image_path),
                'original_file_name': record['file_name'],
                'image_id': int(record['image_id']),
                'original_width': orig_w,
                'original_height': orig_h,
                'yolo_label': yolo_label,
                'coco_category_id': coco_category_id,
                'category_name': category_name,
                'score': float(score.item()),
                'raw_box_original_xyxy': [float(x) for x in box.tolist()],
                'preprocessing': prep_meta,
            },
        )
        manifest_rows.append(meta)
        image_folder_metadata.append(meta) # Hozzáadjuk a mappa-szintű listához is
    
    if len(image_folder_metadata) > 0:
        folder_json_path = image_output_dir / 'annotations.json'
        with folder_json_path.open('w', encoding='utf-8') as f:
            json.dump(image_folder_metadata, f, indent=2)

    # Save annotated teeth missed by YOLO.
    '''
    missed_annotations = find_missed_annotations(
        record['annotations'],
        pred_boxes_original,
        pred_labels,
    )

    for j, ann in enumerate(missed_annotations):
        gt_xyxy = annotation_to_xyxy(ann)
        if gt_xyxy is None:
            continue

        crop_box = expand_and_clip_xyxy(gt_xyxy, orig_w, orig_h, margin=cfg.missed_gt_crop_margin)
        if crop_box is None:
            continue

        coco_category_id = int(ann['category_id'])
        category_name = record['id_to_name'].get(coco_category_id, str(coco_category_id))
        crop_name = f'missed_gt_{j:03d}_ann_{int(ann.get("id", -1))}_class_{coco_category_id}'

        meta = save_crop_with_metadata(
            orig_img=orig_img,
            output_image_dir=image_output_dir,
            crop_box=crop_box,
            crop_name=crop_name,
            metadata={
                'source': 'missed_ground_truth_annotation',
                'run_id': RUN_ID,
                'split': split,
                'original_image_path': str(image_path),
                'original_file_name': record['file_name'],
                'image_id': int(record['image_id']),
                'annotation_id': int(ann.get('id', -1)),
                'original_width': orig_w,
                'original_height': orig_h,
                'coco_category_id': coco_category_id,
                'category_name': category_name,
                'score': None,
                'raw_box_original_xyxy': [float(x) for x in gt_xyxy],
                'margin_used': cfg.missed_gt_crop_margin,
                'matched_yolo_iou_threshold': cfg.missed_match_iou_threshold,
            },
        )
        manifest_rows.append(meta)
'''
manifest_df = pd.DataFrame(manifest_rows)
manifest_csv_path = RUN_OUTPUT_DIR / 'manifest.csv'
manifest_jsonl_path = RUN_OUTPUT_DIR / 'manifest.jsonl'

manifest_df.to_csv(manifest_csv_path, index=False)
with manifest_jsonl_path.open('w', encoding='utf-8') as f:
    for row in manifest_rows:
        f.write(json.dumps(row) + '\n')

print('Done.')
print('Saved crops under:', RUN_OUTPUT_DIR)
print('Manifest CSV:', manifest_csv_path)
print('Manifest JSONL:', manifest_jsonl_path)
print('Total saved crops:', len(manifest_df))
display(manifest_df.head())


Processing filtered classification images:   0%|          | 0/5 [00:00<?, ?it/s]

Done.
Saved crops under: /work/scripts/detection_pipeline/inference pipelines/dataprep/prepared_tooth_crops/4ef020aad4
Manifest CSV: /work/scripts/detection_pipeline/inference pipelines/dataprep/prepared_tooth_crops/4ef020aad4/manifest.csv
Manifest JSONL: /work/scripts/detection_pipeline/inference pipelines/dataprep/prepared_tooth_crops/4ef020aad4/manifest.jsonl
Total saved crops: 123


,source,run_id,split,original_image_path,original_file_name,...,preprocessing,crop_path,crop_box_xyxy,crop_width,crop_height
0,yolo_two_stage,4ef020aad4,train,/work/data/classification_dataset/train/cate2-...,cate2-00098_jpg.rf.84a05e227522684a3fbc382976e...,...,"{'orig_w': 640, 'orig_h': 640, 'zoom_box': (32...",/work/scripts/detection_pipeline/inference pip...,"[431, 336, 489, 489]",58,153
1,yolo_two_stage,4ef020aad4,train,/work/data/classification_dataset/train/cate2-...,cate2-00098_jpg.rf.84a05e227522684a3fbc382976e...,...,"{'orig_w': 640, 'orig_h': 640, 'zoom_box': (32...",/work/scripts/detection_pipeline/inference pip...,"[470, 327, 536, 460]",66,133
2,yolo_two_stage,4ef020aad4,train,/work/data/classification_dataset/train/cate2-...,cate2-00098_jpg.rf.84a05e227522684a3fbc382976e...,...,"{'orig_w': 640, 'orig_h': 640, 'zoom_box': (32...",/work/scripts/detection_pipeline/inference pip...,"[515, 326, 565, 447]",50,121
3,yolo_two_stage,4ef020aad4,train,/work/data/classification_dataset/train/cate2-...,cate2-00098_jpg.rf.84a05e227522684a3fbc382976e...,...,"{'orig_w': 640, 'orig_h': 640, 'zoom_box': (32...",/work/scripts/detection_pipeline/inference pip...,"[125, 336, 189, 472]",64,136
4,yolo_two_stage,4ef020aad4,train,/work/data/classification_dataset/train/cate2-...,cate2-00098_jpg.rf.84a05e227522684a3fbc382976e...,...,"{'orig_w': 640, 'orig_h': 640, 'zoom_box': (32...",/work/scripts/detection_pipeline/inference pip...,"[382, 160, 407, 345]",25,185


## 11. Summarize outputs

In [70]:
if len(manifest_rows) == 0:
    print('No crops were saved. Check filtering labels, input paths, and model output thresholds.')
else:
    summary = manifest_df.groupby(['split', 'source']).size().reset_index(name='count')
    display(summary)

    class_summary = manifest_df.groupby(['split', 'source', 'coco_category_id', 'category_name']).size().reset_index(name='count')
    display(class_summary.sort_values(['split', 'source', 'coco_category_id']).head(100))


,split,source,count
0,train,yolo_two_stage,123


,split,source,coco_category_id,category_name,count
0,train,yolo_two_stage,1,3M ESPE Implant,3
1,train,yolo_two_stage,2,47,4
2,train,yolo_two_stage,3,48,4
3,train,yolo_two_stage,4,49,4
4,train,yolo_two_stage,5,50,4
5,train,yolo_two_stage,6,51,3
6,train,yolo_two_stage,7,52,5
7,train,yolo_two_stage,8,53,4
8,train,yolo_two_stage,9,54,3
9,train,yolo_two_stage,10,55,3
